In [6]:
import numpy as np
import torch
from torch import nn
import pyspiel

In [1]:
GAME_DEF = """universal_poker(
    betting=nolimit,
    bettingAbstraction=fullgame,
    numPlayers=6,
    blind=2 1 0 0 0 0,
    numRounds=4,
    firstPlayer=2 1 1 1,
    numSuits=4,
    numRanks=13,
    numHoleCards=2,
    numBoardCards=0 3 1 1,
    stack=200 200 200 200 200 200
)
"""
GAME_DEF = GAME_DEF.replace("    ", "").replace("\n", "")
GAME_DEF

'universal_poker(betting=nolimit,bettingAbstraction=fullgame,numPlayers=6,blind=2 1 0 0 0 0,numRounds=4,firstPlayer=2 1 1 1,numSuits=4,numRanks=13,numHoleCards=2,numBoardCards=0 3 1 1,stack=200 200 200 200 200 200)'

In [19]:
class ResNet(nn.Module):
    def __init__(
            self, embedding_dim, dropout=0.0, prenorm=True, activation=nn.ReLU()):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim),
            activation,
            nn.Dropout(dropout),
        )
        self.layernorm = nn.LayerNorm(embedding_dim)
        self.prenorm = prenorm
        nn.init.trunc_normal_(self.layer[0].weight, std=0.02, a=-0.04, b=0.04)

    def forward(self, x):
        if self.prenorm:
            return x + self.layer(self.layernorm(x))

        return self.layernorm(x + self.layer(x))


class RNadModel(nn.Module):
    def __init__(self, infostate_tensor_shape, num_actions, hidden_dim, dropout):
        super().__init__()

        self.tower = nn.Sequential(
            nn.Linear(infostate_tensor_shape, hidden_dim),
            ResNet(hidden_dim, dropout),
            ResNet(hidden_dim, dropout),
            ResNet(hidden_dim, dropout),
        )
        self.policy_tower = nn.Linear(hidden_dim, num_actions)

        self.value_head = nn.Linear(hidden_dim, 1)
        self.log_policy_head = nn.Sequential(
            self.policy_tower,
            nn.LogSoftmax(dim=-1)
        )
        self.policy_head = nn.Sequential(
            self.policy_tower,
            nn.Softmax(dim=-1)
        )


    def forward(self, x):
        embedding = self.tower(x)
        return (
            self.value_head(embedding),
            self.log_policy_head(embedding),
            self.policy_head(embedding)
        )


game = pyspiel.load_game(GAME_DEF)
infostate_tensor_shape = game.information_state_tensor_shape()[0]
num_actions = game.num_distinct_actions()
model = RNadModel(
    infostate_tensor_shape=infostate_tensor_shape,
    num_actions=num_actions,
    hidden_dim=256,
    dropout=0.1
)

state = game.new_initial_state()
while state.is_chance_node():
    actions, probs = zip(*state.chance_outcomes())
    sampled_action = np.random.choice(actions, p=probs)
    state.apply_action(sampled_action)

infromation_state_tensor = torch.tensor(state.information_state_tensor())
model(infromation_state_tensor)

(tensor([0.1689], grad_fn=<ViewBackward0>),
 tensor([-5.4316, -5.8718, -5.1533, -4.9006, -5.5781, -4.8689, -5.6482, -5.3546,
         -5.0888, -5.6103, -5.2001, -5.0913, -5.3271, -5.5729, -5.0606, -5.4820,
         -5.3785, -5.0711, -5.6298, -5.7091, -5.1633, -5.0098, -5.3988, -5.5859,
         -5.0879, -4.9328, -5.0747, -5.3400, -5.3586, -5.4940, -5.4581, -5.5916,
         -5.3410, -5.2310, -5.0826, -5.2160, -5.4019, -5.0101, -5.1538, -5.4064,
         -5.4485, -5.0613, -5.2442, -5.3075, -5.4908, -5.2087, -5.5111, -5.5681,
         -5.5095, -5.2160, -5.5723, -5.4660, -5.2498, -5.6083, -5.1374, -6.0123,
         -5.4221, -5.2104, -5.3746, -5.7602, -5.0241, -4.9150, -5.6994, -5.5347,
         -5.7585, -5.8742, -5.4801, -5.2385, -5.3834, -5.1190, -5.2415, -5.5370,
         -5.3522, -4.9721, -5.5463, -5.6895, -5.3152, -5.2608, -5.4852, -5.1284,
         -5.2323, -5.2334, -5.6198, -5.8434, -5.0659, -5.3782, -5.3011, -5.5596,
         -5.2815, -4.6436, -5.0274, -5.3110, -4.9402, -5.7166, -5

In [20]:
torch.jit.script(model).save("model.pt")